# Aligned AR with unsupervised scratch grids — 5 slots, trained k = 0, 2, 3 (seed 42)

**Why.** Supervising every intermediate grid (notebook 43) solves this setting, but it shows the model each single
operation applied to the input, so single operations are no longer held out. Here the intermediate grids are
**not supervised** and the model never sees them. It only gets *positions* for them, and the loss is on the final
answer only.

**Model.** The same sequence as notebook 43:

`<BOS> <OPS> [5 slots] <GRID> [36 input] <OUTPUT> [scratch for step 1] … [scratch for step 4] [final grid] <EOS>`

* Each scratch position holds one learned scratch vector (no digit), plus the **step = op-slot embedding of slot s**
  and the (row, col) of its cell. These are the same position encodings as notebook 43.
* There is no loss on the scratch positions and nothing is fed back into them. They are free "thinking space",
  computed in parallel, and the whole model is trained end to end by ordinary backpropagation (no RL).
* Only the final grid is predicted, autoregressively, with target-cell coordinates and a 10-way head. The loss is on
  its 36 digits only.

Initialization is identical to Aligned AR. The test records are the same as in notebook 42 (Aligned AR without
scratch: single operations 0.00, k = 4 0.00, k = 5 0.002); notebook 43 with supervised grids reached 1.00 / 1.00 /
0.9998.

**Send back `AlignedAR_scratch_S5_k023_Review.ipynb`.**


In [ ]:
# --- 2. Bootstrap: locate or clone the PRIVATE repository (credentials are never printed or stored) ---
import base64, importlib.util, os, pathlib, subprocess, sys

REPO_URL = "https://github.com/zhexuandliu/Decompositionality.git"
REPO_BRANCH = "main"
IN_COLAB = "google.colab" in sys.modules or "COLAB_RELEASE_TAG" in os.environ

def _find_repo():
    candidates = [pathlib.Path("/content/Decompositionality"), pathlib.Path.cwd(), *pathlib.Path.cwd().parents]
    for p in candidates:
        if (p / "src" / "decompositionality").is_dir():
            return p.resolve()
    return None

REPO_DIR = _find_repo()
if REPO_DIR is None:
    token = None
    if IN_COLAB:
        try:
            from google.colab import userdata
            token = userdata.get("GITHUB_TOKEN")
        except Exception:
            token = None
    if not token:
        import getpass
        token = getpass.getpass("GitHub token with read access to the private repo (input hidden): ")
    # Pass the credential through environment-scoped git config (not the URL, not the command line,
    # not persisted in .git/config).
    _hdr = "AUTHORIZATION: basic " + base64.b64encode(f"x-access-token:{token}".encode()).decode()
    _env = dict(os.environ, GIT_TERMINAL_PROMPT="0", GIT_CONFIG_COUNT="1",
                GIT_CONFIG_KEY_0="http.https://github.com/.extraheader", GIT_CONFIG_VALUE_0=_hdr)
    REPO_DIR = pathlib.Path("/content/Decompositionality" if IN_COLAB else "Decompositionality").resolve()
    _r = subprocess.run(["git", "clone", "--depth", "1", "--branch", REPO_BRANCH, REPO_URL, str(REPO_DIR)],
                        env=_env, capture_output=True, text=True)
    del token, _hdr, _env
    if _r.returncode != 0:
        raise RuntimeError("git clone failed (check the GITHUB_TOKEN secret / repo access):\n" + _r.stderr[-2000:])

if str(REPO_DIR / "src") not in sys.path:
    sys.path.insert(0, str(REPO_DIR / "src"))
for _mod in ("numpy", "pandas", "matplotlib", "torch"):
    if importlib.util.find_spec(_mod) is None:  # Colab already ships these; never reinstall torch/CUDA here
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", _mod], check=True)
REPO_COMMIT = subprocess.run(["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip()
print("Repository:", REPO_DIR, "| commit:", REPO_COMMIT or "unknown")


In [ ]:
# --- CONFIGURATION -------------------------------------------------------------------------------------------------
MODE = os.environ.get("DECOMP_MODE", "full")
SEED = 42
RUN_CONDITIONS = ['S5_k023']
RUN_MODELS = ["ar_scratch_slotstep"]
STEPS = 20000
SUBDIR = "S7_support_sweep_20k_v1"                # same folder as notebooks 23 / 27 (model subfolders differ)
EVAL_PLAN = {"n_single_grids": 1000, "n_seen_per_k": 5000, "n_long_per_k": 20000}
USE_DRIVE = IN_COLAB
RESULTS_ROOT = os.environ.get("DECOMP_RESULTS_ROOT") or (
    "/content/drive/MyDrive/decompositionality_results" if USE_DRIVE else str(REPO_DIR / "results"))
DEVICE = os.environ.get("DECOMP_DEVICE", "auto")
# -----------------------------------------------------------------------------------------------------------------
if USE_DRIVE and IN_COLAB and RESULTS_ROOT.startswith("/content/drive"):
    from google.colab import drive
    drive.mount("/content/drive")
import pandas as pd
from IPython.display import display
pd.set_option("display.max_rows", 500); pd.set_option("display.max_columns", 60); pd.set_option("display.width", 250)
from decompositionality import longk as LK, longk_exp as LX
print("results ->", LX.family_root(RESULTS_ROOT, MODE, SUBDIR))


In [3]:
# --- Train (20,000 updates) + sampled LAST-checkpoint evaluation (resumable) ---------------------------------------
status = LX.run_queue(RESULTS_ROOT, MODE, SEED, RUN_CONDITIONS, RUN_MODELS, repo_dir=str(REPO_DIR), device=DEVICE,
                      subdir=SUBDIR, steps=STEPS if MODE == "full" else None, sampled_plan=EVAL_PLAN)
display(status)


[S5_k023] S=5 train k=[0, 2, 3] | val 900 | manifests bdf24c327a65

=== S5_k023 / ar_scratch_slotstep ===
  microbatch probe: 256 fits
[ar_scratch_slotstep] 10,670,602 params | device NVIDIA RTX PRO 6000 Blackwell Server Edition | dtype bf16 | microbatch 256 x accumulation 1 = 256
  [val @ 250] cell CE 1.5508 (best) | subset(96) exact 0.000 cell 0.261
  step 350/20000 loss 1.0284 lr 3.00e-04 | 16.95 steps/s, 4338 ex/s | ETA 19.3 min
  [val @ 500] cell CE 0.6365 (best) | subset(96) exact 0.365 cell 0.595
  step 700/20000 loss 0.0834 lr 3.00e-04 | 17.20 steps/s, 4403 ex/s | ETA 18.7 min
  [val @ 750] cell CE 0.0374 (best) | subset(96) exact 0.719 cell 0.975
  [val @ 1000] cell CE 0.0118 (best) | subset(96) exact 0.885 cell 0.995
  step 1050/20000 loss 0.0132 lr 2.99e-04 | 17.07 steps/s, 4369 ex/s | ETA 18.5 min
  [val @ 1250] cell CE 0.0081 (best) | subset(96) exact 0.938 cell 0.998
  step 1400/20000 loss 0.0022 lr 2.98e-04 | 17.13 steps/s, 4386 ex/s | ETA 18.1 min
  [val @ 1500] cell CE

,condition,model,trained,evaluated_last,records,best_step (reference),complete
0,S5_k023,ar_scratch_slotstep,True,True,66000,8000,True


In [4]:
# --- Comparison: no scratch (42), supervised grids (43), unsupervised scratch (this notebook) -------------------------
from decompositionality import shortcut_analysis as SA
models = ["ar_slotpos_digithead_target", "ar_trace_slotstep", "ar_scratch_slotstep"]
df = LX.per_k_accuracy(RESULTS_ROOT, MODE, [SEED], ["S5_k023"], SUBDIR, models)
ok = df[df.status == "available"]
print("Exact accuracy of the final grid (rows: model, columns: test k; k = 1 = single operations; trained k = 0, 2, 3):")
display(ok.pivot_table(index="model", columns="k", values="exact_acc").round(4))
rp = SA.analyse(RESULTS_ROOT, MODE, SEED, conditions=["S5_k023"], models=models,
                review_name="AlignedAR_scratch_S5_k023_Review.ipynb", out_name=f"seed_{SEED}_scratch_S5_k023")
print("REVIEW FILE TO SEND BACK:", rp)
if IN_COLAB and rp:
    from google.colab import files
    files.download(str(rp))


Exact accuracy of the final grid (rows: model, columns: test k; k = 1 = single operations; trained k = 0, 2, 3):


k,0,1,2,3,4,5
model,,,,,,
ar_scratch_slotstep,1.0,0.0075,1.0,1.0,0.0,0.0540
ar_slotpos_digithead_target,1.0,0.0000,1.0,1.0,0.0,0.0019
ar_trace_slotstep,1.0,1.0000,1.0,1.0,1.0,0.9998


# What do the models compute on held-out inputs? (seed 42, LAST checkpoint)
Read-only: rebuilt sampled test sets are checked record-by-record against the saved predictions. A transform is written as its shortest A/B/C word (A = right shift, B = clockwise rotation, C = left-right mirror; executed left to right). 'seen' = the transform is the overall transform of some trained instruction.


### Availability


,condition,model,status
0,S5_k023,AR34,ok
1,S5_k023,Aligned AR + grids (slot step),ok
2,S5_k023,Aligned AR + scratch (no step loss),ok


**Which operation is dropped?** For held-out longer instructions whose prediction equals the word with exactly ONE operation removed, the position (1 = first, k = last) of that operation. 'unique' counts only records where exactly one position fits; 'any' counts every fitting position.

... (11 outputs left out)

,condition,model,k,records,correct,identity copy,= one op dropped,of which = last op dropped,other legal transform,no legal / invalid
0,S5_k023,AR34,4,20000,0.0,0.0,0.382,0.143,0.382,0.618
1,S5_k023,Aligned AR + grids (slot step),4,4000,1.0,0.0,0.000,0.000,0.000,0.000
2,S5_k023,Aligned AR + scratch (no step loss),4,20000,0.0,0.0,0.625,0.250,0.625,0.375


REVIEW FILE: /content/drive/MyDrive/decompositionality_results/S7_support_sweep_20k_v1/report/seed_42_scratch_S5_k023/AlignedAR_scratch_S5_k023_Review.ipynb
REVIEW FILE TO SEND BACK: /content/drive/MyDrive/decompositionality_results/S7_support_sweep_20k_v1/report/seed_42_scratch_S5_k023/AlignedAR_scratch_S5_k023_Review.ipynb


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [6]:
from google.colab import runtime

runtime.unassign()

RuntimeManagementError: Unable to request VM unassignment.